In [ ]:
import pandas as pd
import numpy as np

# --- paths (adjust if needed) ---
csv_path = "./results/accuracy_acyclic/joblight/lpbound_joblight_full_estimations.csv"      # QueryID,lpbound_Estimate
jsonl_path = "./est/joblight/lpbound-10::joblight-queries.jsonl"  # {"tag": "...", "est": ...}

# --- load first file (CSV) ---
df_a = pd.read_csv(csv_path)
df_a = df_a.rename(columns={
    "QueryID": "query_id",
    "lpbound_Estimate": "est [theirs]"
})

# --- load second file (JSONL) ---
df_b = pd.read_json(jsonl_path, lines=True)
df_b["query_id"] = df_b["tag"].astype(int)
df_b = df_b.drop(columns=["tag"]).rename(columns={"lpbound-10": "est [ours]"})

# --- merge ---
df = (
    df_a.merge(df_b, on="query_id", how="inner")
         .sort_values("query_id")
         .reset_index(drop=True)
)

# --- comparisons ---
df["abs_diff"] = (df["est [ours]"] - df["est [theirs]"]).abs()
df["rel_diff"] = df["abs_diff"] / df["est [theirs]"].replace(0, np.nan)
df["equal"] = np.isclose(df["est [theirs]"], df["est [ours]"], rtol=1e-9, atol=1e-9)

# --- convenience flags ---
df["changed"] = ~df["equal"]

# --- show summary ---
display(df)

print("Summary:")
print(df[["changed"]].value_counts())

print("\nTop 10 largest absolute differences:")
display(df.sort_values("abs_diff", ascending=False).head(10))

print("\nTop 10 largest relative differences:")
display(df.sort_values("rel_diff", ascending=False).head(10))
